# Breit-Wigner convolved with detector resolution

This notebook demonstrates the generic one-dimensional convolution layer using the classic example of a Breit-Wigner mass distribution smeared by a Gaussian detector response. The resulting line shape is a Voigt-type profile. This is conceptually distinct from Square-Dalitz SCF migration.


In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BreitWigner1D, ConvolvedPDF1D, FactorizedDensity, GaussianResolution1D,
    Parameter, enable_x64,
)

enable_x64()


## True Breit-Wigner and Gaussian detector response

We use the normalized constant-width Breit-Wigner PDF

$$f(m)\propto \frac{\Gamma/2}{(m-m_0)^2+(\Gamma/2)^2},$$

and a Gaussian resolution kernel

$$R(m_{\rm rec}\mid m)=\frac{1}{\sqrt{2\pi}\sigma}\exp\left[-\frac{(m_{\rm rec}-m)^2}{2\sigma^2}\right].$$

The observed density is

$$g(m_{\rm rec})=\frac{\int f(m)R(m_{\rm rec}\mid m)dm}{\int_{m_{\min}}^{m_{\max}}dm_{\rm rec}\int f(m)R(m_{\rm rec}\mid m)dm}.$$

The denominator is important because a finite fit window can lose probability through the Gaussian tails.


In [ ]:
true_mass = BreitWigner1D(
    mean=5.279, width=0.030, low=5.15, high=5.40,
)
resolution = GaussianResolution1D(sigma=0.012)
reco_mass = ConvolvedPDF1D(
    true_mass, resolution,
    true_low=5.15, true_high=5.40,
    observed_low=5.15, observed_high=5.40,
    order=96,
)

mass = jnp.linspace(5.15, 5.40, 1600)
plt.figure(figsize=(8,5))
plt.plot(mass, true_mass(mass), label="true Breit-Wigner")
plt.plot(mass, reco_mass(mass), label="Breit-Wigner ⊗ Gaussian")
plt.xlabel(r"$m$ [GeV]")
plt.ylabel("normalized density")
plt.legend()
plt.show()


In [ ]:
integral = jnp.trapezoid(reco_mass(mass), mass)
print("Observed-window normalization:", float(integral))
print("Fraction retained before renormalization:", float(reco_mass.normalization()))


## Vary the detector resolution

The intrinsic width $\Gamma$ belongs to the physical Breit-Wigner PDF, while $\sigma_{\rm res}$ belongs to the detector response. Keeping them separate is important in a fit. The resolution width and bias may be ordinary numbers or fit `Parameter` objects.


In [ ]:
sigma_res = Parameter("mass_resolution.sigma", 0.012, bounds=(0.002,0.050))
bias_res = Parameter("mass_resolution.bias", 0.0, bounds=(-0.020,0.020))

floating_reco = ConvolvedPDF1D(
    true_mass,
    GaussianResolution1D(sigma=sigma_res, bias=bias_res),
    true_low=5.15, true_high=5.40,
    observed_low=5.15, observed_high=5.40,
    order=96,
)

narrow = floating_reco(mass, {"mass_resolution.sigma":0.006, "mass_resolution.bias":0.0})
nominal = floating_reco(mass, {"mass_resolution.sigma":0.012, "mass_resolution.bias":0.0})
broad = floating_reco(mass, {"mass_resolution.sigma":0.025, "mass_resolution.bias":0.004})

plt.figure(figsize=(8,5))
plt.plot(mass, true_mass(mass), label="true Breit-Wigner")
plt.plot(mass, narrow, label="σres = 6 MeV")
plt.plot(mass, nominal, label="σres = 12 MeV")
plt.plot(mass, broad, label="σres = 25 MeV, bias = 4 MeV")
plt.xlabel(r"$m$ [GeV]")
plt.ylabel("normalized density")
plt.legend()
plt.show()


## Use inside a factorized Dalitz + discriminant density

The convolved PDF follows the same callable interface as the other 1D PDFs, so it can be inserted directly into `FactorizedDensity`.


In [ ]:
example_mass = jnp.asarray([5.250,5.279,5.315])
base_dalitz_density = lambda pars: jnp.asarray([0.2,0.5,0.3])

combined = FactorizedDensity(
    base_density=base_dalitz_density,
    observables={"mass":example_mass},
    pdfs={"mass":reco_mass},
)
print(combined({}))


## Important distinction from the Dalitz amplitude Breit-Wigner

`BreitWigner1D` is a normalized real PDF for an observed one-dimensional variable. It is not the complex relativistic Breit-Wigner amplitude used for an intermediate resonance in an amplitude model. Detector resolution acts on the reconstructed observable/PDF. In general one should not simply convolve each complex Dalitz amplitude with a Gaussian.

Likewise, a Dalitz-resolution or SCF problem is generally two-dimensional and can move events between distant regions of the physical plane. That remains a migration-kernel problem (`SquareDalitzSCFMap` or a future generalized migration operator), not two independent 1D convolutions in $s_{ij}$.
